## **1. Install Dependencies**

In [ ]:
!pip install -q accelerate Evaluate
!pip list

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
Package                                  Version
---------------------------------------- -------------------
absl-py                                  1.4.0
accelerate                               1.13.0
access                                   1.1.10.post3
affine                                   2.4.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.5
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.18.4
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0

## **2. Import dependencies**

In [ ]:
import sympy
import sympy.printing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
import os
import torch
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score, f1_score, precision_score, recall_score,
    precision_recall_fscore_support,
    confusion_matrix, classification_report,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
    AutoConfig,
    TrainerCallback
)
from datasets import Dataset
os.environ["WANDB_DISABLED"] = "true"
from sklearn.utils.class_weight import compute_class_weight

import torch.nn.functional as F
from google.colab import files # Para descargar el archivo

## **3. Seed**

In [ ]:
# Fijar semilla global
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## **4. Train**

## **4.1 Train functions**

In [ ]:
def run_train_experiment_test(
    train_csv_path,
    test_csv_path,
    model_name,
    labels_to_keep,
    save_model=True,
    max_length=512,
    lr=5e-5,
    batch_size=8,
    num_epochs=10,
    num_freeze_layers=6,
    use_class_weights=False,
    model_output_dir="./trained_model",
    csv_sep=";",
    dropout_prob=0.1,
    weight_decay=0.01,
    warmup_ratio=0.0,
    lr_scheduler_type="linear",
    early_stopping_patience=None
):

    print("\nLoading dataset...")
    train_df = pd.read_csv(train_csv_path)

    # ==================== 1. Preparación de etiquetas ==============================
    train_df = train_df.rename(columns={"label": "class", "requirement_text": "RequirementText"})
    train_df = train_df[train_df["class"].isin(labels_to_keep)].copy()

    print("\n----- Train Class Distribution -----")
    for cls, count in train_df["class"].value_counts().items():
        print(f"{cls:<10} : {count}")

    label2id = {label: idx for idx, label in enumerate(sorted(train_df["class"].unique()))}
    id2label = {idx: label for label, idx in label2id.items()}
    train_df["label"] = train_df["class"].map(label2id)

    # ==================== 2. Split Train / Validation ==============================
    train_df, val_df = train_test_split(
        train_df, test_size=0.10, stratify=train_df["label"], random_state=SEED
    )

    # ==================== 2.5 Cargar Test ==========================================
    print("\nLoading test dataset...")
    test_df = pd.read_csv(test_csv_path, sep=csv_sep)
    test_df = test_df.rename(columns={"requirementText": "RequirementText"})
    test_df = test_df[test_df["iso_class"].isin(labels_to_keep)].copy()
    test_df["label"] = test_df["iso_class"].map(label2id)

    print("\nSplit:")
    print("Train:", len(train_df))
    print("Validation:", len(val_df))
    print("Test:", len(test_df))

    # ==================== 3. Tokenizer ====================
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_function(examples):
        return tokenizer(
            examples["RequirementText"], padding="max_length", truncation=True, max_length=max_length
        )

    train_dataset = Dataset.from_pandas(train_df)
    val_dataset = Dataset.from_pandas(val_df)
    test_dataset = Dataset.from_pandas(test_df)

    tokenized_train = train_dataset.map(tokenize_function, batched=True)
    tokenized_val = val_dataset.map(tokenize_function, batched=True)
    tokenized_test = test_dataset.map(tokenize_function, batched=True)

    # ==================== 4. Métricas ====================
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)

        return {
            "accuracy": accuracy_score(labels, predictions),
            "f1": f1_score(labels, predictions, average="weighted", zero_division=0),
            "precision": precision_score(labels, predictions, average="weighted", zero_division=0),
            "recall": recall_score(labels, predictions, average="weighted", zero_division=0),
        }

    # ==================== 5. Trainer Personalizado ====================
    class WeightedTrainer(Trainer):
        def __init__(self, class_weights=None, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.class_weights = class_weights

        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.get("labels")
            outputs = model(**inputs)
            logits = outputs.get("logits")

            if self.class_weights is not None:
                loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(model.device))
                loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
            else:
                loss = outputs["loss"]

            return (loss, outputs) if return_outputs else loss

        def log(self, logs: dict, *args, **kwargs) -> None:
            if "eval_val_loss" in logs:
                logs["eval_loss"] = logs["eval_val_loss"]

            super().log(logs, *args, **kwargs)

    # ==================== 6. Class Weights ====================
    current_class_weights = None
    if use_class_weights:
        weights = compute_class_weight(
            class_weight="balanced", classes=np.arange(len(label2id)), y=train_df["label"].values
        )
        current_class_weights = torch.tensor(weights, dtype=torch.float32)
        print("\nClass weights enabled")

    # ==================== 7. Modelo ====================
    config = AutoConfig.from_pretrained(
        model_name,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id,
        hidden_dropout_prob=dropout_prob,
        attention_probs_dropout_prob=dropout_prob
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, config=config
    )

    layers = model.base_model.encoder.layer
    print("\nFreezing layers:", num_freeze_layers)

    for layer in layers[:num_freeze_layers]:
        for param in layer.parameters():
            param.requires_grad = False

    # ==================== 8. Training Arguments ====================
    training_args = TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        lr_scheduler_type=lr_scheduler_type,
        load_best_model_at_end=True,
        metric_for_best_model="eval_val_f1",
        greater_is_better=True,
        report_to="none",
        save_total_limit=1
    )

    # ==================== 9. Trainer y Callbacks ====================
    callbacks = []
    if early_stopping_patience is not None:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=early_stopping_patience))

    trainer = WeightedTrainer(
        class_weights=current_class_weights,
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        # === IMPORTANTE: Volvemos a pasar el diccionario ===
        eval_dataset={"val": tokenized_val, "test": tokenized_test},
        compute_metrics=compute_metrics,
        callbacks=callbacks
    )

    # ==================== 10. Entrenamiento ====================
    print("\nStarting Training...")
    trainer.train()

    # (Opcional) Evaluación final detallada del test
    print("\nEvaluating on TEST (PROMISE) dataset with the best model...")
    test_results = trainer.evaluate(eval_dataset=tokenized_test, metric_key_prefix="final_test")

    print("\n--- TEST RESULTS ---")
    for key, value in test_results.items():
        print(f"{key}: {value:.4f}")

    # ==================== 11. Guardar modelo ====================
    if save_model:
        os.makedirs(model_output_dir, exist_ok=True)
        trainer.save_model(model_output_dir)
        tokenizer.save_pretrained(model_output_dir)
        print("\nModel saved in:")
        print(model_output_dir)
    else:
        print("\nModel saving is disabled (save_model=False).")

### binary functions

In [ ]:
def run_train_experiment_test_binary_fr_nfr(
    train_csv_path,
    test_csv_path,
    model_name,
    save_model=True,
    max_length=512,
    lr=5e-5,
    batch_size=8,
    num_epochs=10,
    num_freeze_layers=6,
    use_class_weights=True,
    model_output_dir="./trained_binary_fr_nfr_model",
    csv_sep=",",
    dropout_prob=0.1,
    weight_decay=0.01,
    warmup_ratio=0.0,
    lr_scheduler_type="linear",
    early_stopping_patience=None,
    functional_label="FS",
    nfr_labels=None
):

    print("\nLoading train dataset...")
    train_df = pd.read_csv(train_csv_path)

    # ==================== 1. Configuración binaria ==============================

    if nfr_labels is None:
        nfr_labels = ["PE", "CO", "US", "RE", "SE", "MA", "FL", "SA"]

    labels_to_keep = [functional_label] + nfr_labels

    label2id = {
        "FR": 0,
        "NFR": 1
    }

    id2label = {
        0: "FR",
        1: "NFR"
    }

    def map_iso_to_binary(iso_label):
        iso_label = str(iso_label).strip()

        if iso_label == functional_label:
            return 0

        if iso_label in nfr_labels:
            return 1

        return None

    # ==================== 2. Preparación de TRAIN ==============================

    required_train_columns = ["label", "requirement_text"]
    missing_train_columns = [
        col for col in required_train_columns if col not in train_df.columns
    ]

    if missing_train_columns:
        raise ValueError(
            f"Faltan columnas obligatorias en train CSV: {missing_train_columns}. "
            f"Columnas encontradas: {list(train_df.columns)}"
        )

    train_df = train_df.rename(
        columns={
            "label": "iso_class",
            "requirement_text": "RequirementText"
        }
    )

    train_df["iso_class"] = train_df["iso_class"].astype(str).str.strip()
    train_df["RequirementText"] = train_df["RequirementText"].astype(str)

    train_df = train_df[train_df["iso_class"].isin(labels_to_keep)].copy()
    train_df["label"] = train_df["iso_class"].apply(map_iso_to_binary)

    train_df = train_df.dropna(subset=["label"]).copy()
    train_df["label"] = train_df["label"].astype(int)

    print("\n----- Train Original ISO Distribution -----")
    for cls, count in train_df["iso_class"].value_counts().sort_index().items():
        print(f"{cls:<10} : {count}")

    print("\n----- Train Binary Distribution -----")
    for cls_id, count in train_df["label"].value_counts().sort_index().items():
        print(f"{id2label[cls_id]:<10} : {count}")

    # ==================== 3. Split Train / Validation ==============================

    train_df, val_df = train_test_split(
        train_df,
        test_size=0.10,
        stratify=train_df["label"],
        random_state=SEED
    )

    # ==================== 4. Cargar TEST ==========================================

    print("\nLoading test dataset...")
    test_df = pd.read_csv(test_csv_path, sep=csv_sep)

    # ----------------------------------------------------------------
    # Alias para que funcione con PROMISE y también con tus CSV propios.
    #
    # PROMISE:
    #   requirementText
    #   iso_class
    #
    # Tus CSV:
    #   requirement_text
    #   label
    # ----------------------------------------------------------------

    if "requirementText" in test_df.columns:
        test_text_col = "requirementText"
    elif "requirement_text" in test_df.columns:
        test_text_col = "requirement_text"
    elif "RequirementText" in test_df.columns:
        test_text_col = "RequirementText"
    else:
        raise ValueError(
            "No se encontró columna de texto en test CSV. "
            "Se esperaba una de: requirementText, requirement_text, RequirementText. "
            f"Columnas encontradas: {list(test_df.columns)}"
        )

    if "iso_class" in test_df.columns:
        test_label_col = "iso_class"
    elif "label" in test_df.columns:
        test_label_col = "label"
    elif "class" in test_df.columns:
        test_label_col = "class"
    else:
        raise ValueError(
            "No se encontró columna de clase en test CSV. "
            "Se esperaba una de: iso_class, label, class. "
            f"Columnas encontradas: {list(test_df.columns)}"
        )

    test_df = test_df.rename(
        columns={
            test_text_col: "RequirementText",
            test_label_col: "iso_class"
        }
    )

    test_df["iso_class"] = test_df["iso_class"].astype(str).str.strip()
    test_df["RequirementText"] = test_df["RequirementText"].astype(str)

    test_df = test_df[test_df["iso_class"].isin(labels_to_keep)].copy()
    test_df["label"] = test_df["iso_class"].apply(map_iso_to_binary)

    test_df = test_df.dropna(subset=["label"]).copy()
    test_df["label"] = test_df["label"].astype(int)

    print("\n----- Test Original ISO Distribution -----")
    for cls, count in test_df["iso_class"].value_counts().sort_index().items():
        print(f"{cls:<10} : {count}")

    print("\n----- Test Binary Distribution -----")
    for cls_id, count in test_df["label"].value_counts().sort_index().items():
        print(f"{id2label[cls_id]:<10} : {count}")

    print("\nSplit:")
    print("Train:", len(train_df))
    print("Validation:", len(val_df))
    print("Test:", len(test_df))

    # ==================== 5. Tokenizer ====================

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_function(examples):
        return tokenizer(
            examples["RequirementText"],
            padding="max_length",
            truncation=True,
            max_length=max_length
        )

    train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
    val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
    test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

    tokenized_train = train_dataset.map(tokenize_function, batched=True)
    tokenized_val = val_dataset.map(tokenize_function, batched=True)
    tokenized_test = test_dataset.map(tokenize_function, batched=True)

    # ==================== 6. Métricas ====================

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)

        results = {
            "accuracy": accuracy_score(labels, predictions),

            # En binary, positive class = NFR = 1
            "f1": f1_score(labels, predictions, average="binary", zero_division=0),
            "precision": precision_score(labels, predictions, average="binary", zero_division=0),
            "recall": recall_score(labels, predictions, average="binary", zero_division=0),

            # Más importantes si hay desbalance FR/NFR
            "f1_macro": f1_score(labels, predictions, average="macro", zero_division=0),
            "precision_macro": precision_score(labels, predictions, average="macro", zero_division=0),
            "recall_macro": recall_score(labels, predictions, average="macro", zero_division=0),

            "f1_weighted": f1_score(labels, predictions, average="weighted", zero_division=0),
            "precision_weighted": precision_score(labels, predictions, average="weighted", zero_division=0),
            "recall_weighted": recall_score(labels, predictions, average="weighted", zero_division=0),
        }

        try:
            probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
            nfr_probs = probs[:, 1]
            results["roc_auc"] = roc_auc_score(labels, nfr_probs)
        except Exception:
            results["roc_auc"] = 0.0

        try:
            tn, fp, fn, tp = confusion_matrix(
                labels,
                predictions,
                labels=[0, 1]
            ).ravel()

            results["tn"] = int(tn)
            results["fp"] = int(fp)
            results["fn"] = int(fn)
            results["tp"] = int(tp)
        except Exception:
            results["tn"] = 0
            results["fp"] = 0
            results["fn"] = 0
            results["tp"] = 0

        return results

    # ==================== 7. Trainer Personalizado ====================

    class WeightedTrainer(Trainer):
        def __init__(self, class_weights=None, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.class_weights = class_weights

        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.get("labels")
            outputs = model(**inputs)
            logits = outputs.get("logits")

            if self.class_weights is not None:
                loss_fct = torch.nn.CrossEntropyLoss(
                    weight=self.class_weights.to(model.device)
                )
                loss = loss_fct(
                    logits.view(-1, self.model.config.num_labels),
                    labels.view(-1)
                )
            else:
                loss = outputs["loss"]

            return (loss, outputs) if return_outputs else loss

        def log(self, logs: dict, *args, **kwargs) -> None:
            if "eval_val_loss" in logs:
                logs["eval_loss"] = logs["eval_val_loss"]

            super().log(logs, *args, **kwargs)

    # ==================== 8. Class Weights ====================

    current_class_weights = None

    if use_class_weights:
        weights = compute_class_weight(
            class_weight="balanced",
            classes=np.arange(2),
            y=train_df["label"].values
        )

        current_class_weights = torch.tensor(weights, dtype=torch.float32)

        print("\nClass weights enabled")
        print(f"FR  weight: {current_class_weights[0]:.4f}")
        print(f"NFR weight: {current_class_weights[1]:.4f}")

    # ==================== 9. Modelo ====================

    config = AutoConfig.from_pretrained(
        model_name,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
        hidden_dropout_prob=dropout_prob,
        attention_probs_dropout_prob=dropout_prob
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        config=config
    )

    print("\nFreezing layers:", num_freeze_layers)

    try:
        layers = model.base_model.encoder.layer

        for layer in layers[:num_freeze_layers]:
            for param in layer.parameters():
                param.requires_grad = False

    except AttributeError:
        print("Advertencia: no se pudo acceder a model.base_model.encoder.layer.")
        print("No se congelaron capas.")

    # ==================== 10. Training Arguments ====================

    training_args = TrainingArguments(
        output_dir="./results_binary_fr_nfr",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        lr_scheduler_type=lr_scheduler_type,
        load_best_model_at_end=True,

        # Usar macro F1 porque FR/NFR está desbalanceado.
        metric_for_best_model="eval_val_f1_macro",
        greater_is_better=True,

        report_to="none",
        save_total_limit=1
    )

    # ==================== 11. Trainer y Callbacks ====================

    callbacks = []

    if early_stopping_patience is not None:
        callbacks.append(
            EarlyStoppingCallback(
                early_stopping_patience=early_stopping_patience
            )
        )

    trainer = WeightedTrainer(
        class_weights=current_class_weights,
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset={
            "val": tokenized_val,
            "test": tokenized_test
        },
        compute_metrics=compute_metrics,
        callbacks=callbacks
    )

    # ==================== 12. Entrenamiento ====================

    print("\nStarting Binary FR/NFR Training...")
    trainer.train()

    # ==================== 13. Evaluación final ====================

    print("\nEvaluating on TEST dataset with the best model...")
    test_results = trainer.evaluate(
        eval_dataset=tokenized_test,
        metric_key_prefix="final_test"
    )

    print("\n--- FINAL BINARY FR/NFR TEST RESULTS ---")
    for key, value in test_results.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

    # ==================== 14. Matriz de confusión ====================

    predictions_output = trainer.predict(tokenized_test)
    logits = predictions_output.predictions
    y_true = predictions_output.label_ids
    y_pred = np.argmax(logits, axis=-1)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print("\nConfusion Matrix:")
    print("Rows = true labels, columns = predicted labels")
    print("Labels order: [FR, NFR]")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["FR", "NFR"],
            zero_division=0
        )
    )

    # ==================== 15. Guardar modelo ====================

    if save_model:
        os.makedirs(model_output_dir, exist_ok=True)
        trainer.save_model(model_output_dir)
        tokenizer.save_pretrained(model_output_dir)

        print("\nModel saved in:")
        print(model_output_dir)
    else:
        print("\nModel saving is disabled (save_model=False).")

    return {
        "trainer": trainer,
        "test_results": test_results,
        "label2id": label2id,
        "id2label": id2label,
        "confusion_matrix": cm,
        "train_size": len(train_df),
        "val_size": len(val_df),
        "test_size": len(test_df)
    }

In [ ]:
def run_test_only_binary_fr_nfr(
    model_dir,
    test_csv_path,
    max_length=64,
    batch_size=32,
    csv_sep=",",
    text_col="RequirementText",
    label_col="iso_class",
    functional_label="FS",
    nfr_labels=None
):
    print("\nLoading binary FR/NFR model...")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)

    if nfr_labels is None:
        nfr_labels = ["PE", "CO", "US", "RE", "SE", "MA", "FL", "SA"]

    labels_to_keep = [functional_label] + nfr_labels

    label2id = {
        "FR": 0,
        "NFR": 1
    }

    id2label = {
        0: "FR",
        1: "NFR"
    }

    def map_iso_to_binary(iso_label):
        iso_label = str(iso_label).strip()

        if iso_label == functional_label:
            return 0

        if iso_label in nfr_labels:
            return 1

        return None

    # ============================================================
    # 1. Cargar dataset de test
    # ============================================================

    print("\nLoading test dataset...")
    test_df = pd.read_csv(test_csv_path, sep=csv_sep)

    # Alias para columna de texto
    if text_col in test_df.columns:
        selected_text_col = text_col
    elif "RequirementText" in test_df.columns:
        selected_text_col = "RequirementText"
    elif "requirementText" in test_df.columns:
        selected_text_col = "requirementText"
    elif "requirement_text" in test_df.columns:
        selected_text_col = "requirement_text"
    else:
        raise ValueError(
            "No se encontró columna de texto. "
            "Se esperaba una de: RequirementText, requirementText, requirement_text. "
            f"Columnas encontradas: {list(test_df.columns)}"
        )

    # Alias para columna de clase
    if label_col in test_df.columns:
        selected_label_col = label_col
    elif "iso_class" in test_df.columns:
        selected_label_col = "iso_class"
    elif "label" in test_df.columns:
        selected_label_col = "label"
    elif "class" in test_df.columns:
        selected_label_col = "class"
    else:
        raise ValueError(
            "No se encontró columna de clase. "
            "Se esperaba una de: iso_class, label, class. "
            f"Columnas encontradas: {list(test_df.columns)}"
        )

    test_df = test_df.rename(
        columns={
            selected_text_col: "RequirementText",
            selected_label_col: "iso_class"
        }
    )

    test_df["RequirementText"] = test_df["RequirementText"].astype(str)
    test_df["iso_class"] = test_df["iso_class"].astype(str).str.strip()

    # Filtrar solo clases conocidas
    test_df = test_df[test_df["iso_class"].isin(labels_to_keep)].copy()

    # Mapear ISO -> binario
    test_df["label"] = test_df["iso_class"].apply(map_iso_to_binary)
    test_df = test_df.dropna(subset=["label"]).copy()
    test_df["label"] = test_df["label"].astype(int)

    print("\n----- Test Original ISO Distribution -----")
    for cls, count in test_df["iso_class"].value_counts().sort_index().items():
        print(f"{cls:<10} : {count}")

    print("\n----- Test Binary Distribution -----")
    for cls_id, count in test_df["label"].value_counts().sort_index().items():
        print(f"{id2label[cls_id]:<10} : {count}")

    print("\nTest size:", len(test_df))

    if len(test_df) == 0:
        raise ValueError("El dataset de test quedó vacío después del mapeo binario.")

    # ============================================================
    # 2. Tokenización
    # ============================================================

    def tokenize_function(examples):
        return tokenizer(
            examples["RequirementText"],
            padding="max_length",
            truncation=True,
            max_length=max_length
        )

    test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))
    tokenized_test = test_dataset.map(tokenize_function, batched=True)

    # ============================================================
    # 3. Métricas binarias
    # ============================================================

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)

        results = {
            "accuracy": accuracy_score(labels, predictions),

            # Clase positiva = NFR = 1
            "f1": f1_score(labels, predictions, average="binary", zero_division=0),
            "precision": precision_score(labels, predictions, average="binary", zero_division=0),
            "recall": recall_score(labels, predictions, average="binary", zero_division=0),

            # Recomendadas si hay desbalance FR/NFR
            "f1_macro": f1_score(labels, predictions, average="macro", zero_division=0),
            "precision_macro": precision_score(labels, predictions, average="macro", zero_division=0),
            "recall_macro": recall_score(labels, predictions, average="macro", zero_division=0),

            "f1_weighted": f1_score(labels, predictions, average="weighted", zero_division=0),
            "precision_weighted": precision_score(labels, predictions, average="weighted", zero_division=0),
            "recall_weighted": recall_score(labels, predictions, average="weighted", zero_division=0),
        }

        # ROC-AUC solo tiene sentido si existen ambas clases en el test
        try:
            if len(np.unique(labels)) == 2:
                probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
                nfr_probs = probs[:, 1]
                results["roc_auc"] = roc_auc_score(labels, nfr_probs)
            else:
                results["roc_auc"] = 0.0
        except Exception:
            results["roc_auc"] = 0.0

        try:
            tn, fp, fn, tp = confusion_matrix(
                labels,
                predictions,
                labels=[0, 1]
            ).ravel()

            results["tn"] = int(tn)
            results["fp"] = int(fp)
            results["fn"] = int(fn)
            results["tp"] = int(tp)
        except Exception:
            results["tn"] = 0
            results["fp"] = 0
            results["fn"] = 0
            results["tp"] = 0

        return results

    # ============================================================
    # 4. Trainer solo para evaluación
    # ============================================================

    training_args = TrainingArguments(
        output_dir="./results_binary_test_only",
        per_device_eval_batch_size=batch_size,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        eval_dataset=tokenized_test,
        compute_metrics=compute_metrics
    )

    # ============================================================
    # 5. Evaluación
    # ============================================================

    print("\nEvaluating binary FR/NFR model...")
    results = trainer.evaluate(metric_key_prefix="test")

    print("\n--- BINARY FR/NFR TEST RESULTS ---")
    for key, value in results.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

    # ============================================================
    # 6. Matriz de confusión y reporte
    # ============================================================

    predictions_output = trainer.predict(tokenized_test)
    logits = predictions_output.predictions
    y_true = predictions_output.label_ids
    y_pred = np.argmax(logits, axis=-1)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print("\nConfusion Matrix:")
    print("Rows = true labels, columns = predicted labels")
    print("Labels order: [FR, NFR]")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=["FR", "NFR"],
            zero_division=0
        )
    )

    return {
        "results": results,
        "confusion_matrix": cm,
        "test_df": test_df,
        "y_true": y_true,
        "y_pred": y_pred
    }

## **4.2 Training Instances**

## train main -54

In [ ]:
# ==========================================
# 1. Instanciar el Entrenamiento (intento 1 con 50 projects)
# ==========================================
run_train_experiment_test(
    train_csv_path="./consensus_class.csv",
    test_csv_path="./promise_cleaned2.csv",
    model_name="microsoft/mpnet-base",
    labels_to_keep= ['FS', 'PE', 'CO', 'US', 'RE', 'SE', 'MA', 'FL'],
    save_model=True,

    # --- Parámetros ajustados ---
    max_length=64,              # Aumentado para no cortar el contexto de PROMISE
    batch_size=64,               # Reducido un poco para soportar max_length=128
    lr=3e-5,                     # Ligeramente más alto porque usamos warmup
    num_epochs=10,               # Le damos más tiempo, el early stopping lo frenará si hace falta
    num_freeze_layers=10,
    use_class_weights=False,

    # --- NUEVA REGULARIZACIÓN ---
    dropout_prob=0.3,            # Aumentado (default es 0.1) para evitar memorización
    weight_decay=0.1,           # Penalización más fuerte a los pesos grandes (default es 0.01)
    warmup_ratio=0.1,            # El 10% de los primeros steps el learning rate sube poco a poco
    lr_scheduler_type="cosine",  # Decaimiento suave en forma de curva coseno
    early_stopping_patience=4,   # Si el eval_val_f1 no mejora en 2 épocas, se detiene y guarda el mejor
    model_output_dir="./mpnet_model_main",
    csv_sep=","
)


Loading dataset...

----- Train Class Distribution -----
FS         : 1885
US         : 1829
PE         : 1826
CO         : 1817
MA         : 1808
RE         : 1787
SE         : 1704
FL         : 1684

Loading test dataset...

Split:
Train: 12906
Validation: 1434
Test: 613


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...


Evaluating on TEST (PROMISE) dataset with the best model...


early stopping required metric_for_best_model, but did not find eval_val_f1 so early stopping is disabled



--- TEST RESULTS ---
final_test_loss: 0.8151
final_test_accuracy: 0.7341
final_test_f1: 0.7321
final_test_precision: 0.7360
final_test_recall: 0.7341
final_test_runtime: 2.2863
final_test_samples_per_second: 268.1150
final_test_steps_per_second: 4.3740
epoch: 10.0000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved in:
./mpnet_model_main


## train main binary -54

In [ ]:
binary_fr_nfr_results = run_train_experiment_test_binary_fr_nfr(
    train_csv_path="./consensus_class.csv",
    test_csv_path="./promise_cleaned2.csv",
    model_name="microsoft/mpnet-base",
    save_model=True,

    max_length=64,
    batch_size=64,
    lr=3e-5,
    num_epochs=10,
    num_freeze_layers=10,
    use_class_weights=True,

    dropout_prob=0.3,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    early_stopping_patience=4,

    csv_sep=",",

    functional_label="FS",
    nfr_labels=["PE", "CO", "US", "RE", "SE", "MA", "FL", "SA"],

    model_output_dir="./mpnet_binary_fr_nfr_model"
)


Loading train dataset...

----- Train Original ISO Distribution -----
CO         : 1817
FL         : 1684
FS         : 1885
MA         : 1808
PE         : 1826
RE         : 1787
SA         : 1282
SE         : 1704
US         : 1829

----- Train Binary Distribution -----
FR         : 1885
NFR        : 13737

Loading test dataset...

----- Test Original ISO Distribution -----
CO         : 38
FL         : 42
FS         : 245
MA         : 14
PE         : 55
RE         : 33
SE         : 72
US         : 114

----- Test Binary Distribution -----
FR         : 245
NFR        : 368

Split:
Train: 14059
Validation: 1563
Test: 613


Map:   0%|          | 0/14059 [00:00<?, ? examples/s]

Map:   0%|          | 0/1563 [00:00<?, ? examples/s]

Map:   0%|          | 0/613 [00:00<?, ? examples/s]


Class weights enabled
FR  weight: 4.1448
NFR weight: 0.5686


...


Evaluating on TEST dataset with the best model...


early stopping required metric_for_best_model, but did not find eval_val_f1_macro so early stopping is disabled



--- FINAL BINARY FR/NFR TEST RESULTS ---
final_test_loss: 0.4157
final_test_accuracy: 0.8467
final_test_f1: 0.8661
final_test_precision: 0.9102
final_test_recall: 0.8261
final_test_f1_macro: 0.8434
final_test_precision_macro: 0.8404
final_test_recall_macro: 0.8518
final_test_f1_weighted: 0.8479
final_test_precision_weighted: 0.8544
final_test_recall_weighted: 0.8467
final_test_roc_auc: 0.0000
final_test_tn: 215
final_test_fp: 30
final_test_fn: 64
final_test_tp: 304
final_test_runtime: 2.2505
final_test_samples_per_second: 272.3870
final_test_steps_per_second: 4.4440
epoch: 10.0000

Confusion Matrix:
Rows = true labels, columns = predicted labels
Labels order: [FR, NFR]
[[215  30]
 [ 64 304]]

Classification Report:
              precision    recall  f1-score   support

          FR       0.77      0.88      0.82       245
         NFR       0.91      0.83      0.87       368

    accuracy                           0.85       613
   macro avg       0.84      0.85      0.84       613
we

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved in:
./mpnet_binary_fr_nfr_model


## train ambiguos

In [ ]:
# ==========================================
# 1. Instanciar el Entrenamiento (intento 1 con 50 projects)
# ==========================================
run_train_experiment_test(
    train_csv_path="./ambiguous_requirements.csv",
    test_csv_path="./promise_cleaned2.csv",
    model_name="microsoft/mpnet-base",
    labels_to_keep= ['FS', 'PE', 'CO', 'US', 'RE', 'SE', 'MA', 'FL'],
    save_model=True,

    # --- Parámetros ajustados ---
    max_length=64,              # Aumentado para no cortar el contexto de PROMISE
    batch_size=64,               # Reducido un poco para soportar max_length=128
    lr=3e-5,                     # Ligeramente más alto porque usamos warmup
    num_epochs=10,               # Le damos más tiempo, el early stopping lo frenará si hace falta
    num_freeze_layers=10,
    use_class_weights=False,

    # --- NUEVA REGULARIZACIÓN ---
    dropout_prob=0.3,            # Aumentado (default es 0.1) para evitar memorización
    weight_decay=0.1,           # Penalización más fuerte a los pesos grandes (default es 0.01)
    warmup_ratio=0.1,            # El 10% de los primeros steps el learning rate sube poco a poco
    lr_scheduler_type="cosine",  # Decaimiento suave en forma de curva coseno
    early_stopping_patience=4,   # Si el eval_val_f1 no mejora en 2 épocas, se detiene y guarda el mejor
    model_output_dir="./mpnet_model_ambiguo",
    csv_sep=","
)


Loading dataset...

----- Train Class Distribution -----
FS         : 431
PE         : 431
RE         : 376
US         : 328
SE         : 241
CO         : 220
FL         : 218
MA         : 204

Loading test dataset...

Split:
Train: 2204
Validation: 245
Test: 613


Map:   0%|          | 0/2204 [00:00<?, ? examples/s]

Map:   0%|          | 0/245 [00:00<?, ? examples/s]

Map:   0%|          | 0/613 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...


Evaluating on TEST (PROMISE) dataset with the best model...


early stopping required metric_for_best_model, but did not find eval_val_f1 so early stopping is disabled



--- TEST RESULTS ---
final_test_loss: 1.4605
final_test_accuracy: 0.5808
final_test_f1: 0.5683
final_test_precision: 0.5772
final_test_recall: 0.5808
final_test_runtime: 2.2092
final_test_samples_per_second: 277.4760
final_test_steps_per_second: 4.5270
epoch: 10.0000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved in:
./mpnet_model_ambiguo


## train consistentes

In [ ]:
# ==========================================
# 1. Instanciar el Entrenamiento (intento 1 con 50 projects)
# ==========================================
run_train_experiment_test(
    train_csv_path="./consistent_requirements.csv",
    test_csv_path="./promise_cleaned2.csv",
    model_name="microsoft/mpnet-base",
    labels_to_keep= ['FS', 'PE', 'CO', 'US', 'RE', 'SE', 'MA', 'FL'],
    save_model=True,

    # --- Parámetros ajustados ---
    max_length=64,              # Aumentado para no cortar el contexto de PROMISE
    batch_size=64,               # Reducido un poco para soportar max_length=128
    lr=3e-5,                     # Ligeramente más alto porque usamos warmup
    num_epochs=10,               # Le damos más tiempo, el early stopping lo frenará si hace falta
    num_freeze_layers=10,
    use_class_weights=False,

    # --- NUEVA REGULARIZACIÓN ---
    dropout_prob=0.3,            # Aumentado (default es 0.1) para evitar memorización
    weight_decay=0.1,           # Penalización más fuerte a los pesos grandes (default es 0.01)
    warmup_ratio=0.1,            # El 10% de los primeros steps el learning rate sube poco a poco
    lr_scheduler_type="cosine",  # Decaimiento suave en forma de curva coseno
    early_stopping_patience=4,   # Si el eval_val_f1 no mejora en 2 épocas, se detiene y guarda el mejor
    model_output_dir="./mpnet_model_consistent",
    csv_sep=","
)


Loading dataset...

----- Train Class Distribution -----
CO         : 1386
MA         : 1381
SE         : 1275
FL         : 1227
US         : 1223
FS         : 1219
RE         : 1179
PE         : 1129

Loading test dataset...

Split:
Train: 9017
Validation: 1002
Test: 613


Map:   0%|          | 0/9017 [00:00<?, ? examples/s]

Map:   0%|          | 0/1002 [00:00<?, ? examples/s]

Map:   0%|          | 0/613 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...


Evaluating on TEST (PROMISE) dataset with the best model...


early stopping required metric_for_best_model, but did not find eval_val_f1 so early stopping is disabled



--- TEST RESULTS ---
final_test_loss: 0.8990
final_test_accuracy: 0.6754
final_test_f1: 0.6753
final_test_precision: 0.6881
final_test_recall: 0.6754
final_test_runtime: 2.2611
final_test_samples_per_second: 271.1040
final_test_steps_per_second: 4.4230
epoch: 10.0000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved in:
./mpnet_model_consistent


## **5. Test**

## **5.1 Test Function**

In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

def run_test_only(
    model_dir,
    test_csv_path,
    labels_to_keep,
    max_length,
    batch_size=32,
    csv_sep=";",
    text_col="requirementText",  # <-- Parámetro por defecto
    label_col="iso_class"        # <-- Parámetro por defecto
):
    print("\nLoading model from:", model_dir)

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)

    label2id = model.config.label2id
    id2label = model.config.id2label

    print("\nDetected labels:")
    for k, v in label2id.items():
        print(f"{k} -> {v}")

    print("\nLoading test dataset...")
    df = pd.read_csv(test_csv_path, sep=csv_sep)

    # Validación: verificar que las columnas existen en el CSV
    if text_col not in df.columns:
        raise KeyError(f"La columna de texto '{text_col}' no se encontró. Columnas disponibles: {df.columns.tolist()}")
    if label_col not in df.columns:
        raise KeyError(f"La columna de etiquetas '{label_col}' no se encontró. Columnas disponibles: {df.columns.tolist()}")

    # Filtrar y mapear usando el parámetro label_col
    df = df[df[label_col].isin(labels_to_keep)].copy()
    df["label"] = df[label_col].map(label2id)

    print("\nTest size:", len(df))
    print("\nClass distribution:\n", df[label_col].value_counts().sort_index())

    # Tokenizar usando el parámetro text_col
    def tokenize_function(examples):
        return tokenizer(
            examples[text_col], padding="max_length", truncation=True, max_length=max_length
        )

    dataset = Dataset.from_pandas(df)
    tokenized_dataset = dataset.map(tokenize_function, batched=True)

    training_args = TrainingArguments(
        output_dir="./test_results", per_device_eval_batch_size=batch_size, report_to="none"
    )

    trainer = Trainer(model=model, args=training_args)

    print("\nRunning evaluation...")
    preds = trainer.predict(tokenized_dataset)
    pred_labels = np.argmax(preds.predictions, axis=1)
    true_labels = df["label"].values

    present_label_ids = sorted(list(set(true_labels)))
    present_target_names = [id2label[i] for i in present_label_ids]

    print("\n===== Classification Report =====\n")
    print(classification_report(
        true_labels, pred_labels, labels=present_label_ids, target_names=present_target_names, zero_division=0
    ))

    print("\n===== Confusion Matrix =====\n")
    cm = confusion_matrix(true_labels, pred_labels, labels=present_label_ids)
    cm_df = pd.DataFrame(cm, index=present_target_names, columns=present_target_names)
    cm_df.index.name = "True\\Pred"

    with pd.option_context("display.max_columns", None, "display.width", 1000):
        print(cm_df)

    acc = accuracy_score(true_labels, pred_labels)
    f1 = f1_score(true_labels, pred_labels, average="weighted", zero_division=0)

    print("\nSummary:")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1-weighted: {f1:.4f}")

## **Test Semantic Deduplication Variants**


### Final main multi

In [ ]:
# ==========================================
# 2. Instanciar el Test / Evaluación
# ==========================================
run_test_only(
    model_dir="./mpnet_model_main",
    test_csv_path="./promise_cleaned2.csv",
    labels_to_keep=['FS', 'PE', 'CO', 'US', 'RE', 'SE', 'MA', 'FL'],
    max_length=64,
    batch_size=32,
    csv_sep=",",
    text_col="RequirementText",
    label_col="iso_class"
)


Loading model from: ./mpnet_model_main


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Detected labels:
CO -> 0
FL -> 1
FS -> 2
MA -> 3
PE -> 4
RE -> 5
SE -> 6
US -> 7

Loading test dataset...

Test size: 613

Class distribution:
 iso_class
CO     38
FL     42
FS    245
MA     14
PE     55
RE     33
SE     72
US    114
Name: count, dtype: int64


Map:   0%|          | 0/613 [00:00<?, ? examples/s]


Running evaluation...



===== Classification Report =====

              precision    recall  f1-score   support

          CO       0.64      0.89      0.75        38
          FL       0.50      0.38      0.43        42
          FS       0.82      0.78      0.80       245
          MA       0.40      0.43      0.41        14
          PE       0.74      0.71      0.72        55
          RE       0.68      0.91      0.78        33
          SE       0.80      0.78      0.79        72
          US       0.69      0.69      0.69       114

    accuracy                           0.73       613
   macro avg       0.66      0.70      0.67       613
weighted avg       0.74      0.73      0.73       613


===== Confusion Matrix =====

           CO  FL   FS  MA  PE  RE  SE  US
True\Pred                                 
CO         34   3    1   0   0   0   0   0
FL          8  16    3   0  11   2   0   2
FS          3   5  190   5   0   5  11  26
MA          0   3    2   6   1   2   0   0
PE          1   1    1  

In [ ]:
# ==========================================
# 2. Instanciar el Test / Evaluación
# ==========================================
run_test_only(
    model_dir="./mpnet_model_main",
    test_csv_path="./respan.csv",
    labels_to_keep=['PE', 'CO', 'US', 'RE', 'SE', 'MA', 'FL', 'FS'],
    max_length=64,
    batch_size=32,
    csv_sep=";",
    text_col="RequirementText",
    label_col="iso_class"
)


Loading model from: ./mpnet_model_main


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Detected labels:
CO -> 0
FL -> 1
FS -> 2
MA -> 3
PE -> 4
RE -> 5
SE -> 6
US -> 7

Loading test dataset...

Test size: 107

Class distribution:
 iso_class
FL     5
FS     5
MA     5
PE    19
RE    16
SE    25
US    32
Name: count, dtype: int64


Map:   0%|          | 0/107 [00:00<?, ? examples/s]


Running evaluation...



===== Classification Report =====

              precision    recall  f1-score   support

          FL       0.33      0.80      0.47         5
          FS       0.00      0.00      0.00         5
          MA       0.25      0.20      0.22         5
          PE       0.88      0.74      0.80        19
          RE       0.85      0.69      0.76        16
          SE       0.88      0.92      0.90        25
          US       0.87      0.84      0.86        32

   micro avg       0.77      0.75      0.76       107
   macro avg       0.58      0.60      0.57       107
weighted avg       0.78      0.75      0.75       107


===== Confusion Matrix =====

           FL  FS  MA  PE  RE  SE  US
True\Pred                            
FL          4   0   1   0   0   0   0
FS          1   0   1   0   1   0   2
MA          3   0   1   0   0   0   1
PE          3   0   0  14   1   0   0
RE          1   1   0   0  11   3   0
SE          0   1   0   0   0  23   1
US          0   0   1   2   0   

### Final main binary

In [ ]:
run_test_only_binary_fr_nfr(
    model_dir="./mpnet_binary_fr_nfr_model",
    test_csv_path="./promise_cleaned2.csv",

    max_length=64,
    batch_size=32,
    csv_sep=",",

    text_col="RequirementText",
    label_col="iso_class",

    functional_label="FS",
    nfr_labels=["PE", "CO", "US", "RE", "SE", "MA", "FL"]
)


Loading binary FR/NFR model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Loading test dataset...

----- Test Original ISO Distribution -----
CO         : 38
FL         : 42
FS         : 245
MA         : 14
PE         : 55
RE         : 33
SE         : 72
US         : 114

----- Test Binary Distribution -----
FR         : 245
NFR        : 368

Test size: 613


Map:   0%|          | 0/613 [00:00<?, ? examples/s]


Evaluating binary FR/NFR model...



--- BINARY FR/NFR TEST RESULTS ---
test_loss: 0.4741
test_model_preparation_time: 0.0123
test_accuracy: 0.8467
test_f1: 0.8661
test_precision: 0.9102
test_recall: 0.8261
test_f1_macro: 0.8434
test_precision_macro: 0.8404
test_recall_macro: 0.8518
test_f1_weighted: 0.8479
test_precision_weighted: 0.8544
test_recall_weighted: 0.8467
test_roc_auc: 0.0000
test_tn: 215
test_fp: 30
test_fn: 64
test_tp: 304
test_runtime: 2.8720
test_samples_per_second: 213.4400
test_steps_per_second: 6.9640

Confusion Matrix:
Rows = true labels, columns = predicted labels
Labels order: [FR, NFR]
[[215  30]
 [ 64 304]]

Classification Report:
              precision    recall  f1-score   support

          FR       0.77      0.88      0.82       245
         NFR       0.91      0.83      0.87       368

    accuracy                           0.85       613
   macro avg       0.84      0.85      0.84       613
weighted avg       0.85      0.85      0.85       613



{'results': {'test_loss': 0.4741312861442566,
  'test_model_preparation_time': 0.0123,
  'test_accuracy': 0.8466557911908646,
  'test_f1': 0.8660968660968661,
  'test_precision': 0.9101796407185628,
  'test_recall': 0.8260869565217391,
  'test_f1_macro': 0.8433537765598834,
  'test_precision_macro': 0.8403944798574893,
  'test_recall_macro': 0.8518189884649512,
  'test_f1_weighted': 0.8479172349824754,
  'test_precision_weighted': 0.8543970488394013,
  'test_recall_weighted': 0.8466557911908646,
  'test_roc_auc': 0.0,
  'test_tn': 215,
  'test_fp': 30,
  'test_fn': 64,
  'test_tp': 304,
  'test_runtime': 2.872,
  'test_samples_per_second': 213.44,
  'test_steps_per_second': 6.964},
 'confusion_matrix': array([[215,  30],
        [ 64, 304]]),
 'test_df':      ID                                    RequirementText class iso_class  \
 0     1  The system shall have a MDI form that allows f...     F        FS   
 1     1  The system shall display Events in a vertical ...     F        FS   

In [ ]:
run_test_only_binary_fr_nfr(
    model_dir="./mpnet_binary_fr_nfr_model",
    test_csv_path="./respan.csv",

    max_length=64,
    batch_size=32,
    csv_sep=";",

    text_col="RequirementText",
    label_col="iso_class",

    functional_label="FS",
    nfr_labels=["PE", "US", "RE", "SE", "MA", "FL"]
)


Loading binary FR/NFR model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Loading test dataset...

----- Test Original ISO Distribution -----
FL         : 5
FS         : 5
MA         : 5
PE         : 19
RE         : 16
SE         : 25
US         : 32

----- Test Binary Distribution -----
FR         : 5
NFR        : 102

Test size: 107


Map:   0%|          | 0/107 [00:00<?, ? examples/s]


Evaluating binary FR/NFR model...



--- BINARY FR/NFR TEST RESULTS ---
test_loss: 0.2251
test_model_preparation_time: 0.0043
test_accuracy: 0.9439
test_f1: 0.9706
test_precision: 0.9706
test_recall: 0.9706
test_f1_macro: 0.6853
test_precision_macro: 0.6853
test_recall_macro: 0.6853
test_f1_weighted: 0.9439
test_precision_weighted: 0.9439
test_recall_weighted: 0.9439
test_roc_auc: 0.0000
test_tn: 2
test_fp: 3
test_fn: 3
test_tp: 99
test_runtime: 0.5625
test_samples_per_second: 190.2080
test_steps_per_second: 7.1110

Confusion Matrix:
Rows = true labels, columns = predicted labels
Labels order: [FR, NFR]
[[ 2  3]
 [ 3 99]]

Classification Report:
              precision    recall  f1-score   support

          FR       0.40      0.40      0.40         5
         NFR       0.97      0.97      0.97       102

    accuracy                           0.94       107
   macro avg       0.69      0.69      0.69       107
weighted avg       0.94      0.94      0.94       107



{'results': {'test_loss': 0.2250593602657318,
  'test_model_preparation_time': 0.0043,
  'test_accuracy': 0.9439252336448598,
  'test_f1': 0.9705882352941176,
  'test_precision': 0.9705882352941176,
  'test_recall': 0.9705882352941176,
  'test_f1_macro': 0.6852941176470588,
  'test_precision_macro': 0.6852941176470588,
  'test_recall_macro': 0.6852941176470588,
  'test_f1_weighted': 0.9439252336448598,
  'test_precision_weighted': 0.9439252336448598,
  'test_recall_weighted': 0.9439252336448598,
  'test_roc_auc': 0.0,
  'test_tn': 2,
  'test_fp': 3,
  'test_fn': 3,
  'test_tp': 99,
  'test_runtime': 0.5625,
  'test_samples_per_second': 190.208,
  'test_steps_per_second': 7.111},
 'confusion_matrix': array([[ 2,  3],
        [ 3, 99]]),
 'test_df':       ID                                    RequirementText iso_class  label
 0      1  All entered data can be read, modified, or del...        RE      1
 1      2  All information provided to the application an...        SE      1
 2      3

### Semantic Ambiguous

In [ ]:
# ==========================================
# 2. Instanciar el Test / Evaluación
# ==========================================
run_test_only(
    model_dir="./mpnet_model_ambiguo",
    test_csv_path="./promise_cleaned2.csv",
    labels_to_keep=['FS', 'PE', 'CO', 'US', 'RE', 'SE', 'MA', 'FL'],
    max_length=64,
    batch_size=32,
    csv_sep=",",
    text_col="RequirementText",
    label_col="iso_class"
)


Loading model from: ./mpnet_model_ambiguo


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Detected labels:
CO -> 0
FL -> 1
FS -> 2
MA -> 3
PE -> 4
RE -> 5
SE -> 6
US -> 7

Loading test dataset...

Test size: 613

Class distribution:
 iso_class
CO     38
FL     42
FS    245
MA     14
PE     55
RE     33
SE     72
US    114
Name: count, dtype: int64


Map:   0%|          | 0/613 [00:00<?, ? examples/s]


Running evaluation...



===== Classification Report =====

              precision    recall  f1-score   support

          CO       0.46      0.50      0.48        38
          FL       0.54      0.17      0.25        42
          FS       0.68      0.62      0.65       245
          MA       0.00      0.00      0.00        14
          PE       0.48      0.55      0.51        55
          RE       0.40      0.58      0.47        33
          SE       0.52      0.74      0.61        72
          US       0.61      0.67      0.64       114

    accuracy                           0.58       613
   macro avg       0.46      0.48      0.45       613
weighted avg       0.58      0.58      0.57       613


===== Confusion Matrix =====

           CO  FL   FS  MA  PE  RE  SE  US
True\Pred                                 
CO         19   2    9   0   2   2   2   2
FL          8   7    7   0  15   2   2   1
FS          3   2  152   0   7  13  32  36
MA          1   1    2   0   2   5   0   3
PE          2   0    7  

In [ ]:
# ==========================================
# 2. Instanciar el Test / Evaluación
# ==========================================
run_test_only(
    model_dir="./mpnet_model_ambiguo",
    test_csv_path="./respan.csv",
    labels_to_keep=['PE', 'CO', 'US', 'RE', 'SE', 'MA', 'FL', 'FS'],
    max_length=64,
    batch_size=32,
    csv_sep=";",
    text_col="RequirementText",
    label_col="iso_class"
)


Loading model from: ./mpnet_model_ambiguo


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Detected labels:
CO -> 0
FL -> 1
FS -> 2
MA -> 3
PE -> 4
RE -> 5
SE -> 6
US -> 7

Loading test dataset...

Test size: 107

Class distribution:
 iso_class
FL     5
FS     5
MA     5
PE    19
RE    16
SE    25
US    32
Name: count, dtype: int64


Map:   0%|          | 0/107 [00:00<?, ? examples/s]


Running evaluation...



===== Classification Report =====

              precision    recall  f1-score   support

          FL       0.33      0.20      0.25         5
          FS       0.00      0.00      0.00         5
          MA       0.00      0.00      0.00         5
          PE       0.78      0.95      0.86        19
          RE       0.75      0.38      0.50        16
          SE       0.89      0.96      0.92        25
          US       0.72      0.81      0.76        32

   micro avg       0.74      0.70      0.72       107
   macro avg       0.50      0.47      0.47       107
weighted avg       0.69      0.70      0.68       107


===== Confusion Matrix =====

           FL  FS  MA  PE  RE  SE  US
True\Pred                            
FL          1   0   0   0   0   0   4
FS          0   0   0   0   2   0   3
MA          2   0   0   0   0   0   2
PE          0   0   0  18   0   0   0
RE          0   2   0   3   6   3   1
SE          0   1   0   0   0  24   0
US          0   2   0   2   0   

### Semantic Consistent

In [ ]:
# ==========================================
# 2. Instanciar el Test / Evaluación
# ==========================================
run_test_only(
    model_dir="./mpnet_model_consistent",
    test_csv_path="./promise_cleaned2.csv",
    labels_to_keep=['FS', 'PE', 'CO', 'US', 'RE', 'SE', 'MA', 'FL'],
    max_length=64,
    batch_size=32,
    csv_sep=",",
    text_col="RequirementText",
    label_col="iso_class"
)


Loading model from: ./mpnet_model_consistent


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Detected labels:
CO -> 0
FL -> 1
FS -> 2
MA -> 3
PE -> 4
RE -> 5
SE -> 6
US -> 7

Loading test dataset...

Test size: 613

Class distribution:
 iso_class
CO     38
FL     42
FS    245
MA     14
PE     55
RE     33
SE     72
US    114
Name: count, dtype: int64


Map:   0%|          | 0/613 [00:00<?, ? examples/s]


Running evaluation...



===== Classification Report =====

              precision    recall  f1-score   support

          CO       0.61      0.89      0.72        38
          FL       0.44      0.29      0.35        42
          FS       0.83      0.70      0.76       245
          MA       0.33      0.43      0.38        14
          PE       0.58      0.67      0.62        55
          RE       0.53      0.58      0.55        33
          SE       0.74      0.78      0.76        72
          US       0.60      0.69      0.64       114

    accuracy                           0.68       613
   macro avg       0.58      0.63      0.60       613
weighted avg       0.69      0.68      0.68       613


===== Confusion Matrix =====

           CO  FL   FS  MA  PE  RE  SE  US
True\Pred                                 
CO         34   3    0   0   0   0   0   1
FL          8  12    3   0  15   2   0   2
FS          3   4  171   5   0   6  15  41
MA          0   3    0   6   1   3   1   0
PE          2   2    0  

In [ ]:
# ==========================================
# 2. Instanciar el Test / Evaluación
# ==========================================
run_test_only(
    model_dir="./mpnet_model_consistent",
    test_csv_path="./respan.csv",
    labels_to_keep=['PE', 'CO', 'US', 'RE', 'SE', 'MA', 'FL', 'FS'],
    max_length=64,
    batch_size=32,
    csv_sep=";",
    text_col="RequirementText",
    label_col="iso_class"
)


Loading model from: ./mpnet_model_consistent


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Detected labels:
CO -> 0
FL -> 1
FS -> 2
MA -> 3
PE -> 4
RE -> 5
SE -> 6
US -> 7

Loading test dataset...

Test size: 107

Class distribution:
 iso_class
FL     5
FS     5
MA     5
PE    19
RE    16
SE    25
US    32
Name: count, dtype: int64


Map:   0%|          | 0/107 [00:00<?, ? examples/s]


Running evaluation...



===== Classification Report =====

              precision    recall  f1-score   support

          FL       0.31      0.80      0.44         5
          FS       0.00      0.00      0.00         5
          MA       0.40      0.40      0.40         5
          PE       0.87      0.68      0.76        19
          RE       0.83      0.62      0.71        16
          SE       0.89      0.96      0.92        25
          US       0.84      0.81      0.83        32

   micro avg       0.76      0.74      0.75       107
   macro avg       0.59      0.61      0.58       107
weighted avg       0.77      0.74      0.74       107


===== Confusion Matrix =====

           FL  FS  MA  PE  RE  SE  US
True\Pred                            
FL          4   0   1   0   0   0   0
FS          1   0   1   0   1   0   2
MA          2   0   2   0   0   0   1
PE          4   0   0  13   1   0   0
RE          1   0   0   1  10   3   1
SE          0   0   0   0   0  24   1
US          1   1   1   1   0   

In [ ]:
# 1. Comprimir todo el contenido del directorio actual (/content)
# Se excluyen explícitamente 'sample_data' y '.config' (carpeta oculta por defecto en Colab)
!zip -r resultados_modelo.zip . -x "sample_data/*" -x ".config/*" -x ".*"

# 2. Importar el módulo de archivos de Colab y descargar el ZIP
from google.colab import files
files.download('resultados_modelo.zip')

  adding: ambiguous_requirements.csv (deflated 72%)
  adding: mpnet_model_ambiguo/ (stored 0%)
  adding: mpnet_model_ambiguo/model.safetensors (deflated 33%)
  adding: mpnet_model_ambiguo/config.json (deflated 51%)
  adding: mpnet_model_ambiguo/tokenizer_config.json (deflated 48%)
  adding: mpnet_model_ambiguo/training_args.bin (deflated 54%)
  adding: mpnet_model_ambiguo/tokenizer.json (deflated 71%)
  adding: mpnet_model_consistent/ (stored 0%)
  adding: mpnet_model_consistent/model.safetensors (deflated 33%)
  adding: mpnet_model_consistent/config.json (deflated 51%)
  adding: mpnet_model_consistent/tokenizer_config.json (deflated 48%)
  adding: mpnet_model_consistent/training_args.bin (deflated 54%)
  adding: mpnet_model_consistent/tokenizer.json (deflated 71%)
  adding: results/ (stored 0%)
  adding: results/checkpoint-1269/ (stored 0%)
  adding: results/checkpoint-1269/rng_state.pth (deflated 26%)
  adding: results/checkpoint-1269/scheduler.pt (deflated 61%)
  adding: results/che

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>